# Chapter 13
## Hopf Bifurcations
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter13.ipynb)

## About this chapter

Nine schematics of the Hopf-bifurcation normal form in polar coordinates,
$\dot r = f(r; I)$, $\dot\theta=1$, covering both the subcritical case
($f=Ir+r^3$, and its quintic-corrected version $f=Ir+r^3-r^5$ that
restores a stable outer cycle) and the supercritical case ($f=Ir-r^3$):
the radial-flow curves themselves, their bifurcation diagrams (oscillation
amplitude vs. $I$), and phase-plane portraits with spiraling trajectories.

See [`README.md`](chapter13.md) for the full guide, including suggested
order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact
from mnd.core import draw_arrow

## Subcritical Radial Flow, $f=Ir+r^3$

In [ ]:
def plot_hopf_sub():
    r = np.arange(101) / 100 * 1.2
    plt.figure(figsize=(7, 7))
    for I in [-1, 0, 1]:
        plt.plot(r, I * r + r ** 3, color='k', linewidth=2)
    plt.plot([0, 1.2], [0, 0], color='k', linestyle='dashed')
    plt.text(0.55, 1.4, '$I=1$', fontsize=16)
    plt.text(0.95, 0.70, '$I=0$', fontsize=16)
    plt.text(0.7, -0.6, '$I=-1$', fontsize=16)
    plt.xlim(0, 1.2)
    plt.ylim(-1, 3)
    plt.xticks(np.arange(0, 1.21, 0.3))
    plt.xlabel('$r$')
    plt.ylabel('$f$')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_hopf_sub()

## Quintic-Corrected Subcritical Radial Flow, $f=Ir+r^3-r^5$

In [ ]:
def plot_hopf_sub_2():
    r = np.arange(101) / 100 * 1.2
    plt.figure(figsize=(7, 7))
    for I in [-0.2, 0, 0.2, -0.4]:
        plt.plot(r, I * r + r ** 3 - r ** 5, color='k', linewidth=2)
    plt.plot([0, 1.2], [0, 0], color='k', linestyle='dashed')
    plt.text(0.7, 0.23, '$I=0$', fontsize=16)
    plt.text(0.6, 0.07, '$I=-0.2$', fontsize=16)
    plt.text(0.8, 0.39, '$I=0.2$', fontsize=16)
    plt.text(0.5, -0.06, '$I=-0.4$', fontsize=16)
    plt.xlim(0, 1.2)
    plt.ylim(-0.5, 0.5)
    plt.xticks(np.arange(0, 1.21, 0.3))
    plt.xlabel('$r$')
    plt.ylabel('$f$')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_hopf_sub_2()

## Subcritical Bifurcation Diagram

In [ ]:
def plot_hopf_sub_bif_diag():
    I = np.arange(-100, 1) / 100
    plt.figure(figsize=(6, 6))
    plt.plot(I, np.zeros(101), color='k', linewidth=2)
    plt.plot(-I, np.zeros(101), color='k', linewidth=2, linestyle='dashed')
    plt.plot(I, np.sqrt(-I), color='k', linewidth=2, linestyle='dashed')
    plt.plot(I, -np.sqrt(-I), color='k', linewidth=2, linestyle='dashed')
    plt.xlim(-1, 1)
    plt.ylim(-1, 1)
    plt.gca().set_box_aspect(1)
    plt.xlabel('$I$')
    plt.ylabel('oscillation amplitude')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_hopf_sub_bif_diag()

## Quintic-Corrected Subcritical Bifurcation Diagram

Restoring the $-r^5$ term brings back a stable outer limit-cycle branch
past the point where the plain subcritical normal form would blow up.

In [ ]:
def plot_hopf_sub_bif_diag_2():
    plt.figure(figsize=(6, 6))

    I = -1 + np.arange(101) / 100 * 0.75
    plt.plot(I, np.zeros(101), color='k', linewidth=2)

    I = -0.25 + np.arange(101) / 100 * 0.25
    r0 = np.sqrt(1 / 2 - np.sqrt(1 / 4 + I))
    R0 = np.sqrt(1 / 2 + np.sqrt(1 / 4 + I))
    plt.plot(I, np.zeros(101), color='k', linewidth=2)
    plt.plot(I, r0, color='k', linewidth=2, linestyle='dashed')
    plt.plot(I, -r0, color='k', linewidth=2, linestyle='dashed')
    plt.plot(I, R0, color='k', linewidth=2)
    plt.plot(I, -R0, color='k', linewidth=2)

    I = np.arange(101) / 100
    R0 = np.sqrt(1 / 2 + np.sqrt(1 / 4 + I))
    plt.plot(I, np.zeros(101), color='k', linewidth=2, linestyle='dashed')
    plt.plot(I, R0, color='k', linewidth=2)
    plt.plot(I, -R0, color='k', linewidth=2)

    plt.xlim(-1, 1)
    plt.ylim(-2, 2)
    plt.gca().set_box_aspect(1)
    plt.xlabel('$I$')
    plt.ylabel('oscillation amplitude')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_hopf_sub_bif_diag_2()

## Subcritical Phase Plane

In [ ]:
def spiral_sub(I, r0, theta0, t_final, dt=0.01, reverse=False):
    """dr/dt = I*r + r^3 (subcritical Hopf normal form radial dynamics).

    reverse=True integrates -f(r) forward with theta decreasing instead --
    traces the same trajectory type (unstable circle / manifold) from the
    other temporal direction, which is numerically stable to compute even
    though the true forward trajectory would take unboundedly long to
    leave the neighborhood of a fixed point.
    """
    sign = -1 if reverse else 1
    m_steps = round(t_final / dt)
    r, theta = np.zeros(m_steps + 1), np.zeros(m_steps + 1)
    r[0], theta[0] = r0, theta0
    for k in range(m_steps):
        r_inc = sign * (I * r[k] + r[k] ** 3)
        r_tmp = r[k] + dt / 2 * r_inc
        r_inc = sign * (I * r_tmp + r_tmp ** 3)
        r[k + 1] = r[k] + dt * r_inc
        theta[k + 1] = theta[k] + sign * dt
    return r * np.cos(theta), r * np.sin(theta)


def plot_hopf_sub_phase_plane():
    A, B, C, D = -.5, .5, -.5, .5

    def panel(ax, I, r0_pairs, title):
        if I < 0:
            ax.plot(0, 0, '.k', markersize=18)
        else:
            ax.plot(0, 0, 'ok', markerfacecolor='none')
        if I < 0:
            th = np.arange(101) / 100 * 2 * np.pi
            ax.plot(np.sqrt(-I) * np.cos(th), np.sqrt(-I) * np.sin(th),
                    color='k', linewidth=1, linestyle='dashed')
        for r0, t_fwd, t_rev, phi in r0_pairs:
            x, y = spiral_sub(I, r0, 0.0, t_fwd)
            ax.plot(x, y, color='k', linewidth=1)
            u = np.array([x[1] - x[0], y[1] - y[0]])
            rot = np.array([[np.cos(phi), np.sin(phi)], [-np.sin(phi), np.cos(phi)]])
            u = rot @ u
            draw_arrow(ax, [A, B], [C, D], x[0], y[0], u, epsilon=0.075, width=1)
            x, y = spiral_sub(I, r0, 0.0, t_rev, reverse=True)
            ax.plot(x, y, color='k', linewidth=1)
        ax.set_xlim(A, B)
        ax.set_ylim(C, D)
        ax.set_box_aspect(1)
        ax.set_xlabel('$x$')
        ax.set_ylabel('$y$')
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    panel(axes[0], I=-0.1, r0_pairs=[(0.1, 5, 10, 0.3), (0.4, 3, 3, 0.1)], title='$I<0$')
    panel(axes[1], I=0.1, r0_pairs=[(0.3, 5, 10, 0.1)], title='$I>0$')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_hopf_sub_phase_plane()

## Quintic-Corrected Subcritical Phase Plane

In [ ]:
def spiral_sub_2(I, r0, theta0, t_final, dt=0.001, reverse=False):
    """dr/dt = I*r + r^3 - r^5 (quintic Hopf normal form radial dynamics)."""
    sign = -1 if reverse else 1
    m_steps = round(t_final / dt)
    r, theta = np.zeros(m_steps + 1), np.zeros(m_steps + 1)
    r[0], theta[0] = r0, theta0
    for k in range(m_steps):
        r_inc = sign * (I * r[k] + r[k] ** 3 - r[k] ** 5)
        r_tmp = r[k] + dt / 2 * r_inc
        r_inc = sign * (I * r_tmp + r_tmp ** 3 - r_tmp ** 5)
        r[k + 1] = r[k] + dt * r_inc
        theta[k + 1] = theta[k] + sign * dt
    return r * np.cos(theta), r * np.sin(theta)


def trajectory_with_arrow(ax, xlim, ylim, I, r0, theta0, t_final, phi, spiral=spiral_sub_2):
    x, y = spiral(I, r0, theta0, t_final)
    ax.plot(x, y, color='k', linewidth=1)
    u = np.array([x[1] - x[0], y[1] - y[0]])
    rot = np.array([[np.cos(phi), np.sin(phi)], [-np.sin(phi), np.cos(phi)]])
    u = rot @ u
    draw_arrow(ax, xlim, ylim, x[0], y[0], u, epsilon=0.075, width=1)
    x, y = spiral(I, r0, theta0, t_final, reverse=True)
    ax.plot(x, y, color='k', linewidth=1)


def plot_hopf_sub_phase_plane_2():
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

    ax = axes[0]
    A, B, C, D = -.5, .5, -.5, .5
    ax.plot(0, 0, '.k', markersize=18)
    trajectory_with_arrow(ax, [A, B], [C, D], I=-0.3, r0=0.2, theta0=0, t_final=5, phi=0.0)
    ax.set_xlim(A, B); ax.set_ylim(C, D); ax.set_box_aspect(1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.set_title(r'$I<-1/4$')

    ax = axes[1]
    I = -0.2
    r0_cycle = np.sqrt(1 / 2 - np.sqrt(1 / 4 + I))
    R0_cycle = np.sqrt(1 / 2 + np.sqrt(1 / 4 + I))
    A, B, C, D = -1.1, 1.1, -1.1, 1.1
    ax.plot(0, 0, '.k', markersize=18)
    th = np.arange(201) / 200 * 2 * np.pi
    ax.plot(r0_cycle * np.cos(th), r0_cycle * np.sin(th), color='k', linewidth=2, linestyle='dashed')
    ax.plot(R0_cycle * np.cos(th), R0_cycle * np.sin(th), color='k', linewidth=3)
    trajectory_with_arrow(ax, [A, B], [C, D], I, r0=0.4, theta0=0, t_final=5, phi=0.2)
    trajectory_with_arrow(ax, [A, B], [C, D], I, r0=0.7, theta0=np.pi / 2, t_final=5, phi=0.2)
    trajectory_with_arrow(ax, [A, B], [C, D], I, r0=1.1, theta0=5 * np.pi / 4, t_final=5, phi=0.0)
    ax.set_xlim(A, B); ax.set_ylim(C, D); ax.set_box_aspect(1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.set_title(r'$-1/4<I<0$')

    ax = axes[2]
    I = 0.1
    R0_cycle = np.sqrt(1 / 2 + np.sqrt(1 / 4 + I))
    A, B, C, D = -1.3, 1.3, -1.3, 1.3
    ax.plot(0, 0, 'ok', markerfacecolor='none')
    ax.plot(R0_cycle * np.cos(th), R0_cycle * np.sin(th), color='k', linewidth=3)
    trajectory_with_arrow(ax, [A, B], [C, D], I, r0=1.3, theta0=3 * np.pi / 4, t_final=5, phi=-0.1)
    trajectory_with_arrow(ax, [A, B], [C, D], I, r0=0.5, theta0=np.pi, t_final=5, phi=0.0)
    ax.set_xlim(A, B); ax.set_ylim(C, D); ax.set_box_aspect(1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.set_title('$I>0$')

    plt.tight_layout()
    plt.show()

In [ ]:
plot_hopf_sub_phase_plane_2()

## Supercritical Radial Flow, $f=Ir-r^3$

In [ ]:
def plot_hopf_sup():
    r = np.arange(101) / 100 * 1.2
    plt.figure(figsize=(7, 7))
    for I in [-1, 0, 1]:
        plt.plot(r, I * r - r ** 3, color='k', linewidth=2)
    plt.plot([0, 1.2], [0, 0], color='k', linestyle='dashed')
    plt.text(0.64, -0.75, '$I=0$', fontsize=16)
    plt.text(0.30, -0.95, '$I=-1$', fontsize=16)
    plt.text(0.94, -0.55, '$I=1$', fontsize=16)
    plt.xlim(0, 1.2)
    plt.ylim(-3, 1)
    plt.xticks(np.arange(0, 1.21, 0.3))
    plt.xlabel('$r$')
    plt.ylabel('$f$')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_hopf_sup()

## Supercritical Bifurcation Diagram

In [ ]:
def plot_hopf_sup_bif_diag():
    I_neg = np.arange(-100, 1) / 100
    I_pos = -I_neg
    plt.figure(figsize=(6, 6))
    plt.plot(I_neg, np.zeros(101), color='k', linewidth=2)
    plt.plot(I_pos, np.zeros(101), color='k', linewidth=2, linestyle='dashed')
    plt.plot(I_pos, np.sqrt(I_pos), color='k', linewidth=2)
    plt.plot(I_pos, -np.sqrt(I_pos), color='k', linewidth=2)
    plt.xlim(-1, 1)
    plt.ylim(-1, 1)
    plt.gca().set_box_aspect(1)
    plt.xlabel('$I$')
    plt.ylabel('oscillation amplitude')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_hopf_sup_bif_diag()

## Supercritical Phase Plane

In [ ]:
def spiral_sup(I, r0, theta0, t_final, dt, reverse=False):
    """dr/dt = I*r - r^3 (supercritical Hopf normal form radial dynamics)."""
    sign = -1 if reverse else 1
    m_steps = round(t_final / dt)
    r, theta = np.zeros(m_steps + 1), np.zeros(m_steps + 1)
    r[0], theta[0] = r0, theta0
    for k in range(m_steps):
        r_inc = sign * (I * r[k] - r[k] ** 3)
        r_tmp = r[k] + dt / 2 * r_inc
        r_inc = sign * (I * r_tmp - r_tmp ** 3)
        r[k + 1] = r[k] + dt * r_inc
        theta[k + 1] = theta0 + sign * (k + 1) * dt
    return r * np.cos(theta), r * np.sin(theta)


def plot_hopf_sup_phase_plane():
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    ax = axes[0]
    A, B, C, D = -.5, .5, -.5, .5
    I = -0.02
    ax.plot(0, 0, '.k', markersize=18)
    x, y = spiral_sup(I, 0.2, 0.0, 5, dt=0.01)
    ax.plot(x, y, color='k', linewidth=1)
    u = np.array([x[1] - x[0], y[1] - y[0]])
    phi = 0.1
    rot = np.array([[np.cos(phi), np.sin(phi)], [-np.sin(phi), np.cos(phi)]])
    u = rot @ u
    draw_arrow(ax, [A, B], [C, D], x[0], y[0], u, epsilon=0.075, width=1)
    x, y = spiral_sup(I, 0.2, 0.0, 5, dt=0.01, reverse=True)
    ax.plot(x, y, color='k', linewidth=1)
    ax.set_xlim(A, B); ax.set_ylim(C, D); ax.set_box_aspect(1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.set_title('$I<0$')

    ax = axes[1]
    A, B, C, D = -2, 2, -2, 2
    I = 0.5
    ax.plot(0, 0, 'ok', markerfacecolor='none')
    th = np.arange(101) / 100 * 2 * np.pi
    ax.plot(np.sqrt(I) * np.cos(th), np.sqrt(I) * np.sin(th), color='k', linewidth=4)

    x, y = spiral_sup(I, 0.4, 0.0, 3, dt=0.001)
    ax.plot(x, y, color='k', linewidth=1)
    u = np.array([x[999] - x[998], y[999] - y[998]])
    phi = 0.3
    rot = np.array([[np.cos(phi), np.sin(phi)], [-np.sin(phi), np.cos(phi)]])
    u = rot @ u
    draw_arrow(ax, [A, B], [C, D], x[999], y[999], u, epsilon=0.075, width=1)
    x, y = spiral_sup(I, 0.4, 0.0, 1, dt=0.01, reverse=True)
    ax.plot(x, y, color='k', linewidth=1)

    x, y = spiral_sup(I, 1.2, -np.pi / 2, 3, dt=0.0001)
    ax.plot(x, y, color='k', linewidth=1)
    u = np.array([x[1] - x[0], y[1] - y[0]])
    draw_arrow(ax, [A, B], [C, D], x[0], y[0], u, epsilon=0.075, width=1)
    x, y = spiral_sup(I, 1.2, -np.pi / 2, 3, dt=0.0001, reverse=True)
    ax.plot(x, y, color='k', linewidth=1)

    ax.set_xlim(A, B); ax.set_ylim(C, D); ax.set_box_aspect(1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.set_title('$I>0$')

    plt.tight_layout()
    plt.show()

In [ ]:
plot_hopf_sup_phase_plane()